<a href="https://colab.research.google.com/github/joaossmacedo/SoccerAnalysis/blob/main/notebooks/analysis/general/xg/overperformance/2025_06_29_xg_overperformance_over_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
# !python3 -m pip install soccerdata
# import soccerdata as sd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
import pandas as pd
pd.set_option('display.max_columns', 100)

# Params

In [3]:
path_shots = '/content/drive/My Drive/database/soccerdata/fbref/raw_clean/shot_events/database.csv'

In [4]:
season = [
  f'{i}{i+1}' for i in range(17, 25)
]

# Code

## Prepare Colab

In [5]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


## Getting data

In [6]:
df_shots = pd.read_csv(path_shots)
df_shots.head(2)

,league,season,game,minute,player,team,xG,PSxG,outcome,distance,body_part,notes,SCA 1_player,SCA 1_event,SCA 2_player,SCA 2_event,is_volley,is_freekick,is_deflected,game_date,game_team_h,game_team_a
0,ENG-Premier League,1718,2017-08-11 Arsenal-Leicester City,2,Alexandre Lacazette,Arsenal,0.06,0.37,Goal,13,Head,NaN,Mohamed Elneny,Pass (Live),Héctor Bellerín,Pass (Live),0,0,0,2017-08-11,Arsenal,Leicester City
1,ENG-Premier League,1718,2017-08-11 Arsenal-Leicester City,4,Riyad Mahrez,Leicester City,0.08,NaN,Off Target,17,Left Foot,NaN,NaN,NaN,NaN,NaN,0,0,0,2017-08-11,Arsenal,Leicester City


## Cleaning data

In [7]:
df_shots['count'] = 0

df_shots['goal'] = (df_shots['outcome'] == 'Goal').astype('int')
df_shots['PSxG'] = df_shots['PSxG'].fillna(0)

df_shots['minute_overtime'] = df_shots['minute'].str.split('+', expand=False).str[1]
df_shots['minute_overtime'] = pd.to_numeric(df_shots['minute_overtime'], errors='coerce')
df_shots.loc[df_shots['minute_overtime'] >= 5, 'minute_overtime'] = '5+'
df_shots['on_overtime'] = ~df_shots['minute_overtime'].isna()

df_shots['minute'] = df_shots['minute'].str.split('+', expand=False).str[0]
df_shots['minute'] = df_shots['minute'].astype('int')
df_shots['minute_cat'] = pd.cut(df_shots['minute'], [0, 15, 30, 45, 60, 75, 90], labels=['0-15', '15-30', '30-45', '45-60', '60-75', '75-90'])

df_shots.head(1)

/tmp/ipython-input-7-1285458466.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '5+' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_shots.loc[df_shots['minute_overtime'] >= 5, 'minute_overtime'] = '5+'


,league,season,game,minute,player,team,xG,PSxG,outcome,distance,body_part,notes,SCA 1_player,SCA 1_event,SCA 2_player,SCA 2_event,is_volley,is_freekick,is_deflected,game_date,game_team_h,game_team_a,count,goal,minute_overtime,on_overtime,minute_cat
0,ENG-Premier League,1718,2017-08-11 Arsenal-Leicester City,2,Alexandre Lacazette,Arsenal,0.06,0.37,Goal,13,Head,NaN,Mohamed Elneny,Pass (Live),Héctor Bellerín,Pass (Live),0,0,0,2017-08-11,Arsenal,Leicester City,0,1,NaN,False,0-15


In [8]:
df_shots['minute_overtime'].value_counts().head(3)

,count
minute_overtime,
1.0,7232
2.0,6363
5+,5822


##  Look at xG overperformance overtime

In [9]:
df_grouped = df_shots.groupby(['minute_cat', 'on_overtime'],observed=True).agg({
    'count': 'count', 'goal': 'sum', 'PSxG': 'sum', 'xG': 'sum'
})

df_grouped['g / xg'] = (df_grouped['goal'] / df_grouped['xG']).apply(lambda x: f"{x:.2%}")
df_grouped['psxg / xg'] = (df_grouped['PSxG'] / df_grouped['xG']).apply(lambda x: f"{x:.2%}")

df_grouped['PSxG'] = df_grouped['PSxG'].astype('int')
df_grouped['xG'] = df_grouped['xG'].astype('int')


df_grouped

count  goal  PSxG    xG  g / xg psxg / xg
minute_cat on_overtime                                           
0-15       False        47383  4918  4898  5104  96.34%    95.96%
15-30      False        53091  5667  5728  5818  97.39%    98.45%
30-45      False        54864  5675  5733  6000  94.58%    95.56%
           True          8529   934   921   975  95.77%    94.52%
45-60      False        58247  6373  6246  6438  98.98%    97.01%
60-75      False        57802  6552  6400  6595  99.33%    97.04%
75-90      False        57991  6457  6437  6666  96.86%    96.56%
           True         20117  2299  2370  2445  93.99%    96.92%

Possibilities for xg underperformance on last 15 minutes of each half:
- shot distribution is different and xg model overvalue some chances that occur more in the end of the halves;
- players are tired, so they make more errors;